# 🛒 Olist Recommender System — Pipeline de Dados

> Este notebook explica as decisões tomadas na construção do pipeline de dados para o Sistema de Recomendação.

**Projeto**: Tech Challenge Fase 02 — Pós-graduação ML Engineering  
**Objetivo**: Documentar as decisões de engenharia de dados de forma narrativa para apresentação.


## 📋 Índice

1. [Escolha do Dataset](#1-dataset)
2. [Merge das Tabelas](#2-merge)
3. [Limpeza do Dataset](#3-limpeza)
4. [Seleção de Features](#4-selecao)
5. [Feature Engineering](#5-fe)
6. [Resultado Final](#6-resultado)


---

## 1. Escolha do Dataset<a id="1-dataset"></a>

### Por que Olist Brazilian E-Commerce?

Escolhemos o dataset **Olist Brazilian E-Commerce** por ser:

| Critério | Requisito do Tech Challenge | Olist Dataset |
|---------|---------------------------|---------------|
| Volume de interações | ≥ 10.000 | **99.785 interações** |
| Granularidade | user-item | customer_unique_id × product_id |
| Features temporais | Necessário para split | `order_purchase_timestamp` disponível |
| Feedback do usuário | Explícito ou implícito | `review_score` (1-5) + histórico |

### Estrutura Original

O dataset é composto por **9 arquivos CSV** relacionais:

```
olist_orders_dataset.csv          — Pedidos (chave central)
olist_order_items_dataset.csv     — Itens dos pedidos
olist_customers_dataset.csv       — Clientes
olist_products_dataset.csv        — Produtos
olist_order_reviews_dataset.csv   — Avaliações
olist_order_payments_dataset.csv  — Pagamentos
olist_sellers_dataset.csv        — Vendedores
olist_geolocation_dataset.csv    — Geolocalização (não usado)
product_category_name_translation.csv — Tradução de categorias
```

### Quantidade de Dados

| Métrica | Valor |
|---------|-------|
| Pedidos totais | ~110.000 |
| Usuários únicos | 93.358 |
| Produtos únicos | 32.216 |
| Categorias | 72 |
| Período | 2016-09-15 a 2018-08-29 |


---

## 2. Merge das Tabelas<a id="2-merge"></a>

### Estratégia de Union

A chave central é `order_id` (pedido). As tabelas são unidas sequencialmente:

```
orders
  └── INNER join → order_items (order_id)
      └── INNER join → customers (customer_id)
          └── INNER join → products (product_id)
              └── LEFT join → reviews (order_id)
                  └── LEFT join → translation (product_category_name)
                      └── LEFT join → payments (order_id, deduped)
```

### Decisões de Design

| Decisão | Justificativa |
|---------|---------------|
| **INNER** para orders-items-customers-products | Garante integridade referencial |
| **LEFT** para reviews | Permite manter interações mesmo sem avaliação |
| **LEFT** para translation | Categorias desconhecidas ficam como 'unknown' |
| **Dedup payments by order_id** | Evita explosão de linhas (1 pedido = N pagamentos) |

### Agregação para User-Item

Após o merge, agregamos no nível **customer_unique_id × product_id**:

```python
groupby(['customer_unique_id', 'product_id', 'product_category_name_english']).agg(
    review_score='mean',
    price='first',
    freight_value='mean',
    payment_type='first',
    order_purchase_timestamp='max',
    has_review='max'
)
```

**Por que agregar?**
- Um usuário pode comprar o mesmo produto múltiplas vezes
- Captura o comportamento consolidado do usuário
- Reduz esparsidade da matriz de interações


---

## 3. Limpeza do Dataset<a id="3-limpeza"></a>

### Filtros Aplicados

| Filtro | Antes | Depois | Justificativa |
|--------|-------|--------|---------------|
| Apenas `order_status = 'delivered'` | ~110.840 | ~99.785 | Evita recomendar itens cancelados/devolvidos |
| Remoção de nulos em `customer_unique_id` | - | 0 nulos | Identificador é obrigatório |
| Remoção de nulos em `product_id` | - | 0 nulos | Identificador é obrigatório |

### Decisões de Limpeza

| Decisão | Impacto |
|---------|---------|
| **Manter 687 nulos em `review_score`** | Decisão consciente: preserva interações válidas que não têm avaliação ainda |
| **Preencher `product_category_name_english` com 'unknown'** | Mantém integridade sem perder dados |
| **Manter `customer_unique_id` (não `customer_id`)** | Mapeia o perfil consolidado do usuário, não apenas o pedido |

### Cold-Start

> ⚠️ **Atenção**: O filtro de cold-start foi **DESABILITADO** porque reduzia o dataset de 99.785 para apenas 2.656 linhas.

A decisão foi manter todas as interações, mesmo com usuários de apenas 1 interação. Isso é comum em e-commerces reais e será tratado pelo modelo neural com técnicas de embeddings.

---

## 4. Seleção de Features<a id="4-selecao"></a>

### Features Removidas

Durante a análise, identificamos e removemos features com problemas em **duas rodadas**:

**Rodada 1 — Revisão manual (FE inicial):**

| Feature | Problema | Ação |
|---------|----------|------|
| `constante_teste_1/2/3` | Variância zero (valores constantes) | Removida |
| `product_volume_cm3` | Multicolinearidade com dimensões | Removida |
| `order_item_id_max` | Correlação > 0.99 com contagem | Removida |
| `payment_installments_sum` | Sem utilidade preditiva | Removida |

**Rodada 2 — Auditoria Spearman (|ρ| > 0.95):**

| Feature | Correlação | Justificativa |
|---------|-----------|---------------|
| `user_recency_days` | ρ = −0.986 com `days_since_reference` | Proxies temporais redundantes |
| `freight_value_log` | ρ = +0.966 com `user_avg_freight` | Ambos medem frete; agregação de usuário é mais robusta |

> Relatório completo: `reports/feature_audit_spearman.md`
> **Resultado surpreendente:** a remoção NÃO melhorou NDCG do modelo com aux features (−13.2%). Production segue sendo `Ablation_FINAL_no_aux_emb32` (sem aux features).

### Features Mantidas (10 → 42 colunas)

| Categoria | Qtd | Exemplos |
|-----------|-----|----------|
| Identificadores | 5 | customer_unique_id, product_id, user_id |
| Target/Sinal | 3 | review_score, has_review, purchase_count |
| Numéricas | 6 | price_log, price_to_freight_ratio, has_price_outlier |
| Temporais | 8 | purchase_month, is_weekend, days_since_reference |
| Categóricas Encodadas | 8 | category_target_enc, payment_type_* (OHE) |
| Agregações Usuário | 6 | user_total_purchases, user_avg_freight |
| Agregações Produto | 6 | product_popularity, product_avg_review_score |

> **NCF consome:** 18 features numéricas selecionadas + 3 embeddings (user, item, category)


---

## 5. Feature Engineering<a id="5-fe"></a>

### 5.1 Features Numéricas

Transformações para estabilizar distribuições assimétricas:

```python
df['price_log'] = np.log1p(df['price'])
df['freight_value_log'] = np.log1p(df['freight_value'])
df['price_to_freight_ratio'] = df['price'] / (df['freight_value'] + 1)
df['has_price_outlier'] = (df['price'] > df['price'].quantile(0.99)).astype(int)
```

### 5.2 Features Temporais

Extração de componentes temporais para capturar sazonalidade:

| Feature | Descrição |
|---------|-----------|
| `purchase_year/month/day_of_week/hour` | Decomposição temporal |
| `is_weekend` | Flag binária para comportamento weekend |
| `is_holiday_season` | Flag para novembro/dezembro (Black Friday, Natal) |
| `days_since_reference` | Dia sequencial para capturar trends |

### 5.3 Features Categóricas

**Target Encoding** (Bayesian smoothing):
```
category_target_enc = (count * mean + alpha * global_mean) / (count + alpha)
```
Onde alpha = 10 (hiperparâmetro de regularização)

**Frequency Encoding**:
Proporção de interações da categoria no total

**One-Hot Encoding** para `payment_type`:
- credit_card, boleto, voucher, debit_card

### 5.4 Agregações de Usuário

Capturam o histórico consolidado de cada usuário:

| Feature | Descrição |
|---------|-----------|
| `user_total_purchases` | Número total de compras |
| `user_avg_price/freight` | Ticket médio do usuário |
| `user_purchase_span_days` | Tempo entre primeira e última compra |
| `user_recency_days` | Dias desde última compra |
| `user_review_rate` | Proporção de compras avaliadas |

### 5.5 Agregações de Produto

Capturam a popularidade e reputação de cada produto:

| Feature | Descrição |
|---------|-----------|
| `product_popularity` | Total de interações |
| `product_unique_buyers` | Número de clientes distintos |
| `product_avg_review_score` | Nota média do produto |
| `product_avg_price/freight` | Preço/frete médios |
| `product_review_rate` | Proporção de vendas com review |


---

## 6. Resultado Final<a id="6-resultado"></a>

### Dataset Processado

| Métrica | Valor |
|---------|-------|
| Shape | 99.785 × 42 |
| Usuários únicos | 93.358 |
| Produtos únicos | 32.216 |
| Sparsity | 99,9967% |
| Período | 2016-09-15 a 2018-08-29 |

### Pipeline DVC

```
Raw CSVs (9 arquivos)
    ↓
prepare (data_preparation.py)
    ↓ interactions.parquet (10 colunas)
featurize (feature_engineering.py)
    ↓ interactions_fe.parquet (42 colunas)
validate (shape check)
    ↓ ready for modeling
train (train.py)
    ↓ MLflow tracking
```

### Próximos Passos

1. **Split Temporal**: 70% treino / 15% validação / 15% teste
2. **Negative Sampling**: 1-4 negativos por positivo para BPR Loss
3. **Modelagem NCF**: Neural Collaborative Filtering com PyTorch
4. **Avaliação**: NDCG@K, Recall@K, MAP@K, Hit Rate@K

---

**Notebooks Complementares**:
- `01_eda.ipynb` — Análise exploratória do interactions.parquet
- `02_eda_feature_engineered.ipynb` — Análise das 42 features
- `demo.ipynb` — Demonstração dos baselines no MLflow
